# 10 — Fine-tune DistilGPT-2 (D1 yes/no, binary toxic)

**Goal:** zero-shot DistilGPT-2 fails; a **small** supervised D1 adaptation makes the same decoder usable.

- Prompt: D1 (`Answer:`) + continuation ` yes` / ` no`
- Loss: causal LM **only on answer tokens**
- Default probe: 8k train / 1k val, 2 epochs → `models/distilgpt2-d1-ft/`

## Colab steps (RAR upload)
1. Runtime → Change runtime type → **GPU** (T4 is fine)
2. Upload your `.rar` (repo or folder that contains `src/`) into `/content`
3. Run all cells — setup extracts the RAR, finds `src/paths.py`, installs deps, downloads data if needed
4. Train (~10–25 min on T4) → zip downloads `distilgpt2-d1-ft.zip`
5. On your laptop, extract into `DeepProjectHate/models/distilgpt2-d1-ft/`
6. Local eval:
```bash
python scripts/run_model_comparison.py --models toxic-bert distilgpt2-d1 distilgpt2-d1-ft --eval-n 200
python scripts/run_fairness.py --models toxic-bert distilgpt2-d1 distilgpt2-d1-ft
```


In [ ]:
# === Colab setup: extract uploaded .rar/.zip, then find repo root ===
import os, sys, shutil, subprocess
from pathlib import Path
from typing import Optional

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("IN_COLAB =", IN_COLAB)

CONTENT = Path("/content") if IN_COLAB else Path.cwd()


def find_repo_root(start: Path) -> Optional[Path]:
    """Return dir containing src/paths.py under start."""
    start = start.resolve()
    for cand in (
        start,
        start / "DeepProjectHate",
        CONTENT / "deephate_from_rar",
        CONTENT / "DeepProjectHate",
    ):
        if (cand / "src" / "paths.py").is_file():
            return cand
    for paths_py in start.rglob("paths.py"):
        if paths_py.parent.name == "src":
            return paths_py.parent.parent
    return None


def extract_archive(archive: Path, dest: Path) -> None:
    """Extract .rar/.zip using 7z (preferred) then unrar/unzip fallbacks."""
    dest.mkdir(parents=True, exist_ok=True)
    suffix = archive.suffix.lower()
    errors = []

    # 7z handles RAR4/RAR5 much more reliably than apt unrar-free
    try:
        subprocess.check_call(["7z", "x", f"-o{dest}", "-y", str(archive)])
        print("Extracted with 7z")
        return
    except (subprocess.CalledProcessError, FileNotFoundError) as exc:
        errors.append(f"7z failed: {exc}")

    if suffix == ".zip":
        try:
            shutil.unpack_archive(str(archive), str(dest))
            print("Extracted with shutil (zip)")
            return
        except Exception as exc:  # noqa: BLE001
            errors.append(f"shutil zip failed: {exc}")

    if suffix == ".rar":
        for cmd in (
            ["unrar", "x", "-o+", str(archive), str(dest) + os.sep],
            ["unar", "-f", "-o", str(dest), str(archive)],
        ):
            try:
                subprocess.check_call(cmd)
                print("Extracted with", cmd[0])
                return
            except (subprocess.CalledProcessError, FileNotFoundError) as exc:
                errors.append(f"{cmd[0]} failed: {exc}")

    raise RuntimeError(
        "Could not extract archive. unrar exit 3 often means RAR5 / broken file / "
        "password. Prefer re-uploading as .zip.\n" + "\n".join(errors)
    )


if IN_COLAB:
    %pip install -q transformers datasets scikit-learn pandas pyarrow accelerate

    # p7zip extracts RAR more reliably than default unrar on Colab
    !apt-get -qq update
    !apt-get -qq install -y p7zip-full unrar >/dev/null

    archives = (
        sorted(CONTENT.glob("*.rar"))
        + sorted(CONTENT.glob("*.RAR"))
        + sorted(CONTENT.glob("*.zip"))
        + sorted(CONTENT.glob("*.ZIP"))
    )
    # ignore tool zips we create later
    archives = [a for a in archives if a.name.lower() not in {"distilgpt2-d1-ft.zip", "distilbert-student.zip"}]

    if archives:
        archive = archives[0]
        extract_dir = CONTENT / "deephate_from_rar"
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        print(f"Extracting {archive.name} -> {extract_dir}")
        extract_archive(archive, extract_dir)
    else:
        print("No .rar/.zip found in /content yet.")
        print("Upload DeepProjectHate.rar (or .zip) via the Files panel, then re-run.")

ROOT = find_repo_root(CONTENT)
if ROOT is None:
    ROOT = find_repo_root(Path.cwd())
if ROOT is None:
    raise FileNotFoundError(
        "Could not find src/paths.py after extract. Archive must contain the repo "
        "(folder with src/, scripts/). Try uploading a .zip instead of .rar."
    )

sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("ROOT", ROOT.resolve())


In [ ]:
from device import get_device

print("device:", get_device())
if IN_COLAB and get_device().type != "cuda":
    raise RuntimeError(
        "No GPU detected. Runtime → Change runtime type → GPU, "
        "then Runtime → Restart session, and re-run from the top."
    )

from paths import TRAIN_CSV, SPLIT_INDICES_PATH, LOCAL_DISTILGPT2_DIR

if not TRAIN_CSV.exists():
    print("Downloading Jigsaw...")
    %run scripts/download_jigsaw.py
if not SPLIT_INDICES_PATH.exists():
    print("Making splits...")
    %run scripts/make_splits.py
if not LOCAL_DISTILGPT2_DIR.exists():
    print("Downloading DistilGPT-2...")
    %run scripts/download_distilgpt2.py

print(
    "train.csv:", TRAIN_CSV.exists(),
    "| splits:", SPLIT_INDICES_PATH.exists(),
    "| distilgpt2:", LOCAL_DISTILGPT2_DIR.exists(),
)


In [ ]:
# Full Colab FT probe (submission-quality for §3.7)
MAX_TRAIN = 8000
MAX_VAL = 1000
EPOCHS = 2
BATCH = 8   # drop to 4 if CUDA OOM

cmd = (
    f"python scripts/train_decoder_ft.py --epochs {EPOCHS} --batch-size {BATCH} "
    f"--max-train-samples {MAX_TRAIN} --max-val-samples {MAX_VAL}"
)
print(cmd)
!{cmd}


In [ ]:
import json
from paths import LOCAL_DISTILGPT2_D1_FT_DIR, DECODER_FT_RESULTS_PATH

print("FT dir:", LOCAL_DISTILGPT2_D1_FT_DIR, "exists=", LOCAL_DISTILGPT2_D1_FT_DIR.exists())
assert LOCAL_DISTILGPT2_D1_FT_DIR.exists(), "Training did not write models/distilgpt2-d1-ft/"

cfg_path = LOCAL_DISTILGPT2_D1_FT_DIR / "decoder_ft_config.json"
for p in (cfg_path, DECODER_FT_RESULTS_PATH):
    if p.exists():
        print("---", p)
        print(json.dumps(json.loads(p.read_text(encoding="utf-8")), indent=2)[:2000])

zip_path = Path("/content/distilgpt2-d1-ft.zip") if IN_COLAB else Path("distilgpt2-d1-ft.zip")
!zip -r {zip_path} models/distilgpt2-d1-ft
print("Zip ready:", zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print("Not on Colab — zip written next to cwd.")
